# Wildfire DNN — Leakage-Free Corrected Notebook

**Purpose:** corrected, leakage-free version of the original capstone model.

**Dataset:** `wildfire_dataset.csv`

The held-out test set is isolated before any data-dependent preprocessing.  
For time-series models, chronological splitting is performed **before** scaling and sliding-window construction.  
Validation data are used for early stopping/model selection; the test set is evaluated only after training is complete.

In [ ]:
# Install only if needed in Colab
!pip -q install pandas numpy matplotlib scikit-learn tensorflow imbalanced-learn

import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

from collections import Counter
from imblearn.over_sampling import SMOTE

In [ ]:
DATA_FILE = "wildfire_dataset.csv"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("Please upload", DATA_FILE)
        uploaded = files.upload()
    except Exception as e:
        raise FileNotFoundError(DATA_FILE + " was not found.") from e

assert os.path.exists(DATA_FILE), "Missing file: " + DATA_FILE

In [ ]:
df = pd.read_csv(DATA_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
invalid_ts = int(df["timestamp"].isna().sum())
df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

FEATURES = ["temperature", "humidity", "gas_value", "flame_detected"]
TARGET = "wildfire_detected"
WINDOW_SIZE = 5

df["flame_detected"] = (
    df["flame_detected"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
)
df[TARGET] = (
    df[TARGET].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
)
df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)

print("Rows after cleaning:", len(df))
print("Invalid timestamps removed:", invalid_ts)
print("Overall labels:", Counter(df[TARGET].astype(int)))

In [ ]:
n = len(df)
train_end = int(0.64 * n)
val_end = int(0.80 * n)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

scaler = MinMaxScaler()
scaler.fit(train_df[FEATURES])

def build_flat_windows(part_df, window_size=WINDOW_SIZE):
    X_scaled = scaler.transform(part_df[FEATURES])
    y = part_df[TARGET].astype(int).to_numpy()

    X_windows, y_windows = [], []
    for i in range(window_size, len(part_df)):
        X_windows.append(X_scaled[i-window_size:i].reshape(-1))
        y_windows.append(y[i])

    return np.asarray(X_windows, dtype=np.float32), np.asarray(y_windows, dtype=np.int32)

X_train, y_train = build_flat_windows(train_df)
X_val,   y_val   = build_flat_windows(val_df)
X_test,  y_test  = build_flat_windows(test_df)

print("Before SMOTE")
print("Train:", X_train.shape, Counter(y_train))
print("Val  :", X_val.shape, Counter(y_val))
print("Test :", X_test.shape, Counter(y_test))

smote = SMOTE(random_state=SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("After train-only SMOTE:", X_train_bal.shape, Counter(y_train_bal))

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Input(shape=(X_train_bal.shape[1],)),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(16, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_bal, y_train_bal,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
prob = model.predict(X_test, verbose=0).reshape(-1)
y_pred = (prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy : {acc:.6f}")
print(f"Precision: {prec:.6f}")
print(f"Recall   : {rec:.6f}")
print(f"F1       : {f1:.6f}")
print()
print(classification_report(y_test, y_pred, target_names=["No wildfire", "Wildfire"], zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=["No", "Yes"])
disp.plot(values_format="d")
plt.title("Wildfire — Leakage-Free Held-Out Test Set")
plt.show()

pd.DataFrame({
    "true_label": y_test,
    "predicted_label": y_pred,
    "predicted_probability": prob
}).to_csv("wildfire_leakage_free_test_predictions.csv", index=False)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Wildfire DNN Training")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
model.save("wildfire_classifier_leakage_free.keras")

meta = {
    "features": FEATURES,
    "window_size": WINDOW_SIZE,
    "scaler_min": scaler.min_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "split": {"train": 0.64, "validation": 0.16, "test": 0.20},
    "smote": "training windows only",
    "decision_threshold": 0.5,
    "seed": SEED
}
with open("wildfire_classifier_leakage_free_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
converter._experimental_lower_tensor_list_ops = True
converter.experimental_enable_resource_variables = False
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("wildfire_classifier_leakage_free.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved corrected wildfire artifacts.")